# 📘 Tutorial & Panduan: Training Model Deep Learning di Google Colab
**Dataset:** `tire_demage_20260917_002759`  
**Target Hardware:** NVIDIA T4 GPU  
**Mode Training:** Modular (Step-by-Step / Per-Code)  
**Model Didukung:**
1. ⚡ **YOLO-X / YOLO11-X** (High-Performance Real-Time Object Detection)
2. 🔥 **YOLO-26 / Edge Variant** (Ultra Fast Architecture for Edge Devices)
3. 🎯 **RF-DETR / RT-DETR** (Real-Time Transformer Object Detection)

---
Notebook ini dirancang secara **modular (per-code slice)**. Anda dapat menjalankan cell satu per satu untuk memantau proses, menguji coba sampel data terlebih dahulu, memahami cara kerja setiap tahapan, dan melakukan debugging jika diperlukan.

### 🖥️ Langkah 0: Cek Akses GPU (Runtime Check)
Sebelum memulai training model Deep Learning, kita harus memastikan bahwa runtime Google Colab menggunakan **GPU (Graphics Processing Unit)** seperti **Tesla T4** agar proses training berjalan cepat.

In [ ]:
# Cek apakah GPU tersedia menggunakan PyTorch dan command nvidia-smi
import torch
import subprocess

gpu_available = torch.cuda.is_available()
print('=== STATUS RUNTIME GOOGLE COLAB ===')
if gpu_available:
    device_name = torch.cuda.get_device_name(0)
    print(f'[OK] GPU Terdeteksi: {device_name}')
    print('\nDetail Spesifikasi GPU:')
    try:
        gpu_info = subprocess.check_output('nvidia-smi', shell=True).decode('utf-8')
        print(gpu_info)
    except Exception:
        print('Tidak dapat memanggil nvidia-smi, namun GPU CUDA tetap aktif.')
else:
    print('[WARNING] GPU tidak terdeteksi! Silakan ganti runtime ke T4 GPU via menu: Runtime > Change runtime type > T4 GPU.')


### 📦 Langkah 1: Instalasi Library & Dependensi
Menginstal library utama: `ultralytics` (untuk YOLO & RT-DETR), `onnx` (untuk export model), `pyyaml` (konfigurasi dataset), dan `tqdm` (indikator progress).

In [ ]:
!pip install -q --upgrade ultralytics onnx onnxruntime onnxsim pyyaml requests urllib3 tqdm
import ultralytics
print(f'Ultralytics Version: {ultralytics.__version__}')
ultralytics.checks()


### ⚙️ Langkah 2: Konfigurasi URL & Inisialisasi Direktori Dataset
Menyiapkan folder lokal di Colab (`/content/dataset/images/train`, `/content/dataset/images/val`, dll) dan inisialisasi session HTTP dengan connection pool & auto-retry 5x.

In [ ]:
import os, glob, shutil, time, requests, yaml, json
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

APP_URL = "https://vision.chitraparatama.com"
YOLO_YAML_URL = "https://vision.chitraparatama.com/api/v1/uploads/upload/datasets/tire_demage_20260917_002759/data.yaml"
COCO_JSON_URL = "https://vision.chitraparatama.com/api/v1/uploads/upload/datasets/tire_demage_20260917_002759/annotations_coco.json"
TASKS_JSON_URL = "https://vision.chitraparatama.com/api/v1/uploads/upload/datasets/tire_demage_20260917_002759/label_studio_tasks.json"

# Setup Session Jaringan dengan Auto-Retry
session = requests.Session()
retries = Retry(total=5, backoff_factor=1.5, status_forcelist=[429, 500, 502, 503, 504], raise_on_status=False)
adapter = HTTPAdapter(max_retries=retries, pool_connections=10, pool_maxsize=10)
session.mount('https://', adapter)
session.mount('http://', adapter)

# Inisialisasi struktur direktori YOLO di Colab
base_dir = os.path.abspath('dataset')
train_img_dir = os.path.join(base_dir, 'images', 'train')
val_img_dir = os.path.join(base_dir, 'images', 'val')
train_lbl_dir = os.path.join(base_dir, 'labels', 'train')
val_lbl_dir = os.path.join(base_dir, 'labels', 'val')

for p in [train_img_dir, val_img_dir, train_lbl_dir, val_lbl_dir]:
    os.makedirs(p, exist_ok=True)

print('✓ Direktori dataset siap di:', base_dir)


### 📄 Langkah 3: Unduh & Periksa File `data.yaml`
File `data.yaml` berisi definisi kelas/kategori objek dan pemetaan ID label.

In [ ]:
print(f'Mengunduh data.yaml dari: {YOLO_YAML_URL}')
try:
    r_yaml = session.get(YOLO_YAML_URL, timeout=(10, 60))
    if r_yaml.status_code == 200:
        with open('data.yaml', 'wb') as f:
            f.write(r_yaml.content)
        print('✓ File data.yaml berhasil diunduh!')
    else:
        print(f'Warning: HTTP {r_yaml.status_code} saat mengunduh data.yaml')
except Exception as e:
    print(f'Error: {e}')

# Baca dan periksa pemetaan kelas
with open('data.yaml', 'r') as f:
    data_cfg = yaml.safe_load(f) or {}

raw_names = data_cfg.get('names', {0: 'object'})
if isinstance(raw_names, dict):
    name_to_id = {str(v).lower(): int(k) for k, v in raw_names.items()}
elif isinstance(raw_names, list):
    name_to_id = {str(v).lower(): i for i, v in enumerate(raw_names)}
else:
    name_to_id = {'object': 0}

print(f'Total Kelas Terdaftar: {len(name_to_id)}')
print('Daftar Kelas (ID -> Nama):')
items_to_show = list(raw_names.items() if isinstance(raw_names, dict) else enumerate(raw_names))
for k, v in items_to_show[:10]:
    print(f'  [{k}] {v}')
if len(items_to_show) > 10:
    print(f'  ... dan {len(items_to_show) - 10} kelas lainnya.')


### 📋 Langkah 4: Unduh Metadata Task Anotasi (`tasks.json`)
Mengunduh file metadata anotasi yang berisi informasi bounding box setiap gambar dari server Raray Vision.

In [ ]:
print(f'Mengunduh metadata anotasi dari: {TASKS_JSON_URL}')
tasks = []
try:
    r_tasks = session.get(TASKS_JSON_URL, timeout=(15, 90))
    if r_tasks.status_code == 200:
        tasks = r_tasks.json()
        print(f'✓ Berhasil memuat {len(tasks)} total tasks anotasi!')
        if tasks:
            t0 = tasks[0]
            print('\nPreview Task 0:')
            print('  - File Name      :', t0.get('data', {}).get('original_filename'))
            print('  - Raw Image URL  :', t0.get('data', {}).get('image'))
            print('  - Jumlah Anotasi :', len(t0.get('annotations', [{}])[0].get('result', [])))
    else:
        print(f'Warning: HTTP {r_tasks.status_code} saat mengunduh tasks.json')
except Exception as e:
    print(f'Error: {e}')


### 🧪 Langkah 5: Smoke Test (Uji Coba Unduh 5 Gambar Pertama)
**Langkah ini sangat penting!** Sebelum mendownload ribuan gambar selama puluhan menit, kita lakukan uji coba mengunduh 5 gambar pertama untuk memverifikasi koneksi, perutean proxy terautentikasi, dan status HTTP 200 OK.

In [ ]:
# Fungsi pembantu untuk konversi URL S3 ke proxy publik terautentikasi
def resolve_image_url(raw_url):
    if not raw_url:
        return ''
    if 'is3.cloudhost.id/onechitra/' in raw_url:
        return raw_url.replace('https://is3.cloudhost.id/onechitra/', f'{APP_URL}/api/v1/uploads/').replace('http://is3.cloudhost.id/onechitra/', f'{APP_URL}/api/v1/uploads/')
    elif raw_url.startswith('/api/v1/uploads/'):
        return f'{APP_URL}{raw_url}'
    return raw_url

print('=== SMOKE TEST: UJI COBA UNDUH 5 GAMBAR PERTAMA ===')
test_samples = tasks[:5]
success_cnt = 0
for i, t in enumerate(test_samples):
    orig_url = t.get('data', {}).get('image', '')
    resolved_url = resolve_image_url(orig_url)
    fname = t.get('data', {}).get('original_filename') or os.path.basename(orig_url.split('?')[0]) or f'test_{i}.jpg'
    try:
        res = session.get(resolved_url, timeout=(10, 30))
        if res.status_code == 200:
            success_cnt += 1
            print(f'[{i+1}/5] ✓ {fname} -> HTTP 200 OK ({len(res.content)/1024:.1f} KB)')
        elif res.status_code == 403 and ('is3.cloudhost.id' in orig_url or 'onechitra' in orig_url):
            suffix = orig_url.split('/onechitra/')[-1] if '/onechitra/' in orig_url else orig_url.split('.id/')[-1]
            fallback_url = f'{APP_URL}/api/v1/uploads/{suffix.lstrip("/")}'
            res_fb = session.get(fallback_url, timeout=(10, 30))
            if res_fb.status_code == 200:
                success_cnt += 1
                print(f'[{i+1}/5] ✓ {fname} via Fallback Proxy -> HTTP 200 OK ({len(res_fb.content)/1024:.1f} KB)')
            else:
                print(f'[{i+1}/5] ✗ {fname} -> HTTP {res_fb.status_code}')
        else:
            print(f'[{i+1}/5] ✗ {fname} -> HTTP {res.status_code}')
    except Exception as err:
        print(f'[{i+1}/5] ✗ {fname} -> Error: {err}')

print(f'\nHasil Smoke Test: {success_cnt}/5 gambar berhasil terhubung!')
if success_cnt > 0:
    print('✓ Jalur pengunduhan terverifikasi! Anda dapat melanjutkan ke Langkah 6.')
else:
    print('✗ Gagal mengunduh sampel gambar. Periksa bahwa server Raray Vision aktif.')


### 📥 Langkah 6: Unduh Seluruh Gambar Dataset Secara Paralel
Mengunduh gambar dan membuat file anotasi label YOLO `.txt` (80% Train, 20% Val).

> **Tips Belajar:** Anda dapat mengubah nilai `LIMIT_DOWNLOAD = None` menjadi angka tertentu (contoh: `100` atau `300`) jika ingin melakukan training uji coba cepat tanpa menunggu semua 3.500+ gambar.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# Atur limit gambar jika ingin training cepat (misal: 200), atau None untuk seluruh dataset
LIMIT_DOWNLOAD = None  # Contoh: 200 untuk uji coba, atau None untuk semua gambar

active_tasks = tasks[:LIMIT_DOWNLOAD] if LIMIT_DOWNLOAD else tasks
print(f'Memulai download {len(active_tasks)} gambar dengan 6 worker paralel...')

def download_and_process_item(item_data):
    idx, item = item_data
    img_url = resolve_image_url(item.get('data', {}).get('image', ''))
    if not img_url:
        return
    fname = item.get('data', {}).get('original_filename') or os.path.basename(img_url.split('?')[0]) or f'img_{idx}.jpg'
    if not fname.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.bmp')):
        fname = f'img_{idx}.jpg'
    split = 'val' if (len(active_tasks) > 1 and idx % 5 == 0) else 'train'
    dest_img = os.path.join(base_dir, 'images', split, fname)
    dest_lbl = os.path.join(base_dir, 'labels', split, os.path.splitext(fname)[0] + '.txt')

    for attempt in range(3):
        try:
            res = session.get(img_url, timeout=(15, 60), stream=True)
            if res.status_code == 403 and ('is3.cloudhost.id' in img_url or 'onechitra' in img_url):
                suffix = img_url.split('/onechitra/')[-1] if '/onechitra/' in img_url else img_url.split('.id/')[-1]
                fallback_url = f'{APP_URL}/api/v1/uploads/{suffix.lstrip("/")}'
                res = session.get(fallback_url, timeout=(15, 60), stream=True)

            if res.status_code == 200:
                with open(dest_img, 'wb') as f:
                    for chunk in res.iter_content(chunk_size=65536):
                        if chunk:
                            f.write(chunk)
                lines = []
                for ann in item.get('annotations', [{}])[0].get('result', []):
                    val = ann.get('value', {})
                    lbls = val.get('rectanglelabels', ['object'])
                    first_lbl = str(lbls[0]).lower() if lbls else 'object'
                    cid = name_to_id.get(first_lbl, 0)
                    x_pct = val.get('x', 0) / 100.0
                    y_pct = val.get('y', 0) / 100.0
                    w_pct = val.get('width', 0) / 100.0
                    h_pct = val.get('height', 0) / 100.0
                    xc = x_pct + (w_pct / 2.0)
                    yc = y_pct + (h_pct / 2.0)
                    lines.append(f'{cid} {xc:.6f} {yc:.6f} {w_pct:.6f} {h_pct:.6f}')
                with open(dest_lbl, 'w') as lf:
                    lf.write('\n'.join(lines))
                return
            elif res.status_code == 404:
                return
        except Exception:
            if attempt < 2:
                time.sleep(1 + attempt)

with ThreadPoolExecutor(max_workers=6) as ex:
    list(tqdm(ex.map(download_and_process_item, enumerate(active_tasks)), total=len(active_tasks)))

# Fallback: pastikan train dan val selalu memiliki minimal 1 file gambar & label
train_imgs = glob.glob(os.path.join(train_img_dir, '*.*'))
val_imgs = glob.glob(os.path.join(val_img_dir, '*.*'))
if not val_imgs and train_imgs:
    for f in train_imgs[:max(1, len(train_imgs)//5)]:
        shutil.copy(f, val_img_dir)
        lbl_src = os.path.join(train_lbl_dir, os.path.splitext(os.path.basename(f))[0] + '.txt')
        if os.path.exists(lbl_src): shutil.copy(lbl_src, val_lbl_dir)
elif not train_imgs and val_imgs:
    for f in val_imgs:
        shutil.copy(f, train_img_dir)
        lbl_src = os.path.join(val_lbl_dir, os.path.splitext(os.path.basename(f))[0] + '.txt')
        if os.path.exists(lbl_src): shutil.copy(lbl_src, train_lbl_dir)

# Update data.yaml path ke absolute path dataset
data_cfg['path'] = base_dir
data_cfg['train'] = 'images/train'
data_cfg['val'] = 'images/val'
with open('data.yaml', 'w') as f:
    yaml.dump(data_cfg, f, sort_keys=False)

print('✓ Pengunduhan selesai!')


### 🔍 Langkah 7: Verifikasi & Inspeksi Dataset Lokal
Memeriksa jumlah file gambar dan file label di folder `train` dan `val`, serta memastikan dataset siap dipakai untuk training.

In [ ]:
train_imgs = glob.glob(os.path.join(train_img_dir, '*.*'))
val_imgs = glob.glob(os.path.join(val_img_dir, '*.*'))
train_lbls = glob.glob(os.path.join(train_lbl_dir, '*.txt'))
val_lbls = glob.glob(os.path.join(val_lbl_dir, '*.txt'))

print('=== HASIL PERSIAPAN DATASET LOKAL ===')
print(f'Total Gambar Train : {len(train_imgs)}')
print(f'Total Label Train  : {len(train_lbls)}')
print(f'Total Gambar Val   : {len(val_imgs)}')
print(f'Total Label Val    : {len(val_lbls)}')

if len(train_imgs) == 0:
    raise RuntimeError('Dataset train kosong (0 gambar)! Periksa kembali koneksi atau jalankan ulang Langkah 6.')

# Tampilkan contoh isi file label txt
sample_lbl = train_lbls[0] if train_lbls else None
if sample_lbl:
    print(f'\nContoh Format Label ({os.path.basename(sample_lbl)}):')
    with open(sample_lbl, 'r') as lf:
        print(lf.read().strip())

print('\n✓ Konfigurasi data.yaml akhir:')
!cat data.yaml


### ⚡ Langkah 8: MODEL 1 - Training YOLO-X / YOLO11-X (200 Epochs, T4 GPU)
**YOLO-X** adalah model arsitektur besar untuk akurasi tertinggi dalam deteksi objek real-time.
- **Pretrained Weights:** `yolo11x.pt`
- **Epochs:** `200`
- **Optimizer:** `AdamW`
- **Batch Size:** `16` (Optimal untuk VRAM 16GB Tesla T4)

In [ ]:
from ultralytics import YOLO

print('🚀 MEMULAI TRAINING YOLO-X (200 EPOCHS)...')
model_yolox = YOLO('yolo11x.pt')

results_yolox = model_yolox.train(
    data='data.yaml',
    epochs=200,
    imgsz=640,
    batch=16,
    device=0, # GPU 0 (Tesla T4)
    workers=4,
    optimizer='AdamW',
    lr0=0.001,
    patience=50,
    save=True,
    project='raray_vision_runs',
    name='yolo_x_200epochs'
)

print('\n📊 VALIDASI YOLO-X:')
metrics_yolox = model_yolox.val()
print('mAP50    :', metrics_yolox.box.map50)
print('mAP50-95 :', metrics_yolox.box.map)

# Export ke format ONNX
model_yolox.export(format='onnx', dynamic=True, simplify=True)
print('✓ YOLO-X weights & ONNX berhasil diekspor!')


### 🔥 Langkah 9: MODEL 2 - Training YOLO-26 (Edge Variant, 200 Epochs, T4 GPU)
**YOLO-26** dirancang untuk perangkat Edge (Raspberry Pi, Jetson Nano, Mini PC) dengan FPS tinggi dan konsumsi memori rendah.
- **Pretrained Weights:** `yolo11m.pt`
- **Epochs:** `200`
- **Optimizer:** `SGD`
- **Batch Size:** `24`

In [ ]:
from ultralytics import YOLO

print('🔥 MEMULAI TRAINING YOLO-26 VARIANT (200 EPOCHS)...')
model_yolo26 = YOLO('yolo11m.pt')

results_yolo26 = model_yolo26.train(
    data='data.yaml',
    epochs=200,
    imgsz=640,
    batch=24,
    device=0,
    workers=4,
    optimizer='SGD',
    lr0=0.01,
    patience=50,
    save=True,
    project='raray_vision_runs',
    name='yolo_26_200epochs'
)

print('\n📊 VALIDASI YOLO-26:')
metrics_yolo26 = model_yolo26.val()
print('mAP50    :', metrics_yolo26.box.map50)
print('mAP50-95 :', metrics_yolo26.box.map)

# Export ke ONNX
model_yolo26.export(format='onnx', dynamic=True, simplify=True)
print('✓ YOLO-26 weights & ONNX berhasil diekspor!')


### 🎯 Langkah 10: MODEL 3 - Training RF-DETR / RT-DETR (Transformer, 200 Epochs, T4 GPU)
**RF-DETR / RT-DETR** adalah arsitektur Transformer Vision State-of-the-Art yang tidak membutuhkan Non-Maximum Suppression (NMS), sangat unggul untuk deteksi objek padat dan tumpang tindih.
- **Pretrained Weights:** `rtdetr-l.pt`
- **Epochs:** `200`
- **Optimizer:** `AdamW`
- **Batch Size:** `12`

In [ ]:
from ultralytics import RTDETR

print('🎯 MEMULAI TRAINING RF-DETR / RT-DETR TRANSFORMER (200 EPOCHS)...')
model_rfdetr = RTDETR('rtdetr-l.pt')

results_rfdetr = model_rfdetr.train(
    data='data.yaml',
    epochs=200,
    imgsz=640,
    batch=12,
    device=0,
    workers=4,
    optimizer='AdamW',
    lr0=0.0001,
    patience=50,
    save=True,
    project='raray_vision_runs',
    name='rfdetr_200epochs'
)

print('\n📊 VALIDASI RF-DETR:')
metrics_rfdetr = model_rfdetr.val()
print('Validation Results:', metrics_rfdetr)

# Export ke ONNX
model_rfdetr.export(format='onnx', dynamic=True, simplify=True)
print('✓ RF-DETR weights & ONNX berhasil diekspor!')


### 📦 Langkah 11: Pengemasan Bobot Model & Hasil Evaluasi ke File ZIP
Tahap ini mengumpulkan file bobot terbaik (`best.pt`), bobot ONNX (`best.onnx`), kurva metrik, confusion matrix, dan `results.csv` ke dalam arsip ZIP yang rapi.
Anda dapat menentukan **Tanggal Training** dan **Keterangan Setting / Hyperparameters** agar otomatis tercatat saat diunggah ke sistem Raray Vision.

In [ ]:
# ==========================================================
# PENGATURAN METADATA & ARSIP ZIP TRAINING
# ==========================================================
import os, glob, shutil, json, zipfile
from datetime import datetime

# 📝 Masukkan tanggal dan keterangan setting model Anda:
TANGGAL_TRAINING = datetime.now().strftime('%Y-%m-%d')  # Format: YYYY-MM-DD
KETERANGAN_SETTING = '200 Epochs, Imgsz 640, Batch 16, Optimizer AdamW, Tesla T4 GPU'

print('=== PENGEMASAN ARSIP EVALUASI & BOBOT MODEL ===')
# Cari run training terbaru di raray_vision_runs
all_runs = sorted(glob.glob('raray_vision_runs/*'), key=os.path.getmtime, reverse=True)
# Filter hanya direktori run
valid_runs = [r for r in all_runs if os.path.isdir(r) and not r.endswith('.zip')]

if not valid_runs:
    print('❌ Belum ada folder hasil training di "raray_vision_runs". Silakan jalankan salah satu cell training di Langkah 8, 9, atau 10 terlebih dahulu.')
else:
    latest_run = valid_runs[0]
    run_name = os.path.basename(latest_run)
    print(f'📁 Direktori Training Terpilih: {latest_run}')
    
    # 1. Buat file metadata setting training
    meta_info = {
        'training_date': TANGGAL_TRAINING,
        'training_settings': KETERANGAN_SETTING,
        'run_name': run_name,
        'created_at': datetime.now().isoformat()
    }
    meta_path = os.path.join(latest_run, 'training_meta.json')
    with open(meta_path, 'w', encoding='utf-8') as mf:
        json.dump(meta_info, mf, indent=2)
    print(f'✓ Metadata setting tersimpan di: {meta_path}')
    
    # 2. Kemas Arsip Evaluasi (.zip) untuk diunggah ke Raray Vision
    eval_zip_name = f'evaluasi_{run_name}_{TANGGAL_TRAINING}.zip'
    eval_target_files = [
        'results.csv', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
        'PR_curve.png', 'F1_curve.png', 'results.png', 'labels.jpg',
        'val_batch0_pred.jpg', 'training_meta.json'
    ]
    with zipfile.ZipFile(eval_zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
        for ef in eval_target_files:
            fp = os.path.join(latest_run, ef)
            if os.path.exists(fp):
                zf.write(fp, arcname=ef)
                print(f'  + Arsip Eval: {ef}')
    print(f'✓ File Evaluasi ZIP siap: {eval_zip_name} ({os.path.getsize(eval_zip_name)/1024:.1f} KB)')
    
    # 3. Identifikasi bobot best.pt dan best.onnx
    best_pt_path = os.path.join(latest_run, 'weights', 'best.pt')
    best_onnx_path = os.path.join(latest_run, 'weights', 'best.onnx')
    if not os.path.exists(best_onnx_path):
        pot_onnx = glob.glob(os.path.join(latest_run, '*.onnx'))
        if pot_onnx:
            best_onnx_path = pot_onnx[0]
    
    if os.path.exists(best_pt_path):
        sz_pt = os.path.getsize(best_pt_path) / (1024 * 1024)
        print(f'✓ File Bobot Model: {best_pt_path} ({sz_pt:.2f} MB)')
    else:
        print('⚠ File best.pt belum ditemukan di folder weights.')
        
    # 4. Buat All-In-One ZIP Bundle (Weights + Evaluasi)
    bundle_zip_name = f'raray_vision_{run_name}_{TANGGAL_TRAINING}_bundle.zip'
    with zipfile.ZipFile(bundle_zip_name, 'w', zipfile.ZIP_DEFLATED) as bzf:
        if os.path.exists(best_pt_path):
            bzf.write(best_pt_path, arcname='best.pt')
        if os.path.exists(best_onnx_path):
            bzf.write(best_onnx_path, arcname='best.onnx')
        for ef in eval_target_files:
            fp = os.path.join(latest_run, ef)
            if os.path.exists(fp):
                bzf.write(fp, arcname=ef)
    print(f'✓ File Bundle Lengkap ZIP: {bundle_zip_name} ({os.path.getsize(bundle_zip_name)/(1024*1024):.2f} MB)')


### 📥 Langkah 12: Download Otomatis Bobot (`best.pt`) & Evaluasi (`.zip`) ke Komputer
Menjalankan fungsi download Google Colab ke browser lokal Anda. Setelah terunduh, file siap diunggah ke web **Raray Vision** di menu **Model Management**.

In [ ]:
# ==========================================================
# DOWNLOAD LANGSUNG KE KOMPUTER ANDA
# ==========================================================
try:
    from google.colab import files
    print('📥 Memicu dialog pengunduhan browser...')
    
    if 'best_pt_path' in locals() and os.path.exists(best_pt_path):
        print(f'⬇️ Mengunduh model weights: best.pt ({os.path.getsize(best_pt_path)/(1024*1024):.2f} MB)...')
        files.download(best_pt_path)
    
    if 'eval_zip_name' in locals() and os.path.exists(eval_zip_name):
        print(f'⬇️ Mengunduh arsip evaluasi: {eval_zip_name}...')
        files.download(eval_zip_name)
        
    print('\n🎉 Selesai! File sedang diunduh oleh browser Anda.')
    print('\n📋 PANDUAN UNGGAH KE RARAY VISION:')
    print('1. Buka web Raray Vision > Menu "Model Management".')
    print('2. Klik tombol "+ Upload Model (.pt / .onnx)".')
    print('3. Isi formulir:')
    print('   - File Bobot Model (.pt): Pilih file best.pt yang baru diunduh.')
    print(f'   - File Arsip Evaluasi (.zip): Pilih file {eval_zip_name}.')
    print(f'   - Tanggal Training: {TANGGAL_TRAINING}')
    print(f'   - Keterangan Setting: {KETERANGAN_SETTING}')
    print('4. Klik "Simpan Model". Metrik mAP, Confusion Matrix, dan PR Curve akan langsung tampil!')
except ImportError:
    print('ℹ️ Script tidak dijalankan di Google Colab. File zip dan weights tersedia di direktori lokal.')
